# TOF inspection (multi-file)

Loads one or more combined H5 files, picks one of the zero-padded TOF
hit arrays (`tofs_e`, `tofs_i`, or `liq_tofs_e`), and produces
side-by-side diagnostic plots comparing them.

Per file:

1. Mean spectrum (overlay across files).
2. Mean spectrum vs bunch index — one 2D map per file, plus an
   overlay of integrated hits per bunch.
3. Mean spectrum vs train index — same.
4. GMD vs per-shot hit count — one hist2d per file + overlay of
   binned mean ± std.
5. Mean spectrum per GMD percentile bin — one panel per file
   (lines coloured by GMD bin centre), and a GMD-normalised version
   (each spectrum divided by its bin's mean GMD).

Specify the list in `FILES` (and optionally matching `LABELS`); every
downstream cell loops over a `runs` dict so adding or removing a file
just adds or removes a panel/line. Per-file panels wrap to multiple
rows past `MAX_COLS` columns via the same `_file_grid` helper used in
`cfg1_tr_etof_ion_vs_delay`.

To keep memory in check, the full `(n_trains, n_bunches, n_bins)`
per-shot spectra array is never materialised. The non-zero hits are
flattened once per file and labelled with their train/bunch index,
so each plot is a single `np.histogram` / `np.histogram2d` call.

In [ ]:
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.signal import savgol_filter

import config
from data_loading import load_data
from binning import arb_bool_ar, bin_and_average
%matplotlib inline

## Parameters

`FILES` is the list of combined H5 files to compare. `LABELS` is an
optional matching list; missing entries fall back to each file's
stem.

`CONFIG` and `TOF_KEY` apply to every file. Config 1 has
`tofs_e` / `tofs_i`; config 2 has `liq_tofs_e`.

`MIN_TID` drops trains with `tID` below this threshold (useful for
filtering out empty / dummy trains). Set to `None` to skip.

The TOF histogram grid (in 100 ps units) and the number of GMD
percentile bins are shared across all files. `MAX_COLS` is the wrap
threshold for the multi-file subplot grids.

In [ ]:
# --- input -----------------------------------------------------------
FILES = [
    config.COMBINED_DIR / "glycine_delay_scan_150C_274.0eV.h5",
]
# Optional human-readable labels (one per file). Missing entries
# fall back to the file stem.
LABELS = []

CONFIG  = 1
TOF_KEY = "tofs_i"          # tofs_e | tofs_i | liq_tofs_e

# --- load knobs ------------------------------------------------------
TRIM_START   = 3
TRIM_END     = 4
DOWNSAMPLE_N = 1
MIN_TID      = 1e8          # drop trains with tID < this; None to skip

# --- TOF histogram grid (100 ps) ------------------------------------
TOF_MIN = 0.0
TOF_MAX = 40000.0
N_BINS  = 500

# --- GMD percentile binning -----------------------------------------
N_GMD_BINS = 30

# --- cosmetics ------------------------------------------------------
LOG_MAP   = False
MAX_COLS  = 3               # wrap multi-file panels past this many cols
YLIM_SPEC_BY_GMD = (0, 1e-3)   # ylim for spec-by-GMD plots; None for auto

print(f"files to compare: {len(FILES)}")
for p in FILES:
    print(f"  {p}")

## Helpers

`_label_for` derives a fallback label from each file's stem.

`_file_grid` allocates a `(n_metric_rows-per-file, n_files-cols)`
axes grid that wraps to multiple file-rows past `MAX_COLS` columns;
leftover panels are hidden. Same helper as `cfg1_tr_etof_ion_vs_delay`.

In [ ]:
def _label_for(i, path):
    if i < len(LABELS) and LABELS[i]:
        return LABELS[i]
    return Path(path).stem


def _file_grid(n_metric_rows, n_files, panel_size=(6.0, 4.0),
               max_cols=None, sharey="row", sharex=True):
    if max_cols is None:
        max_cols = MAX_COLS
    cols = min(max_cols, n_files)
    file_rows = (n_files + cols - 1) // cols
    fig, axes = plt.subplots(
        n_metric_rows * file_rows, cols,
        figsize=(panel_size[0] * cols,
                 panel_size[1] * n_metric_rows * file_rows),
        sharey=sharey, sharex=sharex, constrained_layout=True,
        squeeze=False,
    )

    def ax_at(k, m):
        fr, fc = divmod(k, cols)
        return axes[fr * n_metric_rows + m, fc]

    def hide_unused():
        for k in range(n_files, file_rows * cols):
            fr, fc = divmod(k, cols)
            for m in range(n_metric_rows):
                axes[fr * n_metric_rows + m, fc].axis("off")

    return fig, axes, ax_at, hide_unused

## Load and flatten each file

Per file: `load_data` → optional `filter_trains(tID >= MIN_TID)` →
pull the requested TOF array → flatten the non-zero hits and label
each with its train/bunch index. The flat arrays power every
downstream plot via `np.histogram(2d)`.

`runs` is a dict keyed by label; every downstream cell iterates over
`runs.items()`.

In [ ]:
tof_edges = np.linspace(TOF_MIN, TOF_MAX, N_BINS + 1)
tof_cents = 0.5 * (tof_edges[:-1] + tof_edges[1:])


def load_and_prepare(path):
    data = load_data(str(path), config=CONFIG,
                     trim_start=TRIM_START, trim_end=TRIM_END,
                     downsample_N=DOWNSAMPLE_N)
    if MIN_TID is not None:
        data = data.filter_trains(data.tID, lo=MIN_TID, inside=True)

    tofs = getattr(data, TOF_KEY)
    if tofs is None:
        raise ValueError(
            f"{TOF_KEY!r} is not populated for config={CONFIG} in "
            f"{Path(path).name}. Config 1 has tofs_e / tofs_i; "
            f"config 2 has liq_tofs_e."
        )
    n_trains, n_bunches, max_hits = tofs.shape

    valid = tofs != 0
    tof_of_hit   = tofs[valid]
    train_of_hit = np.broadcast_to(
        np.arange(n_trains)[:, None, None], tofs.shape
    )[valid]
    bunch_of_hit = np.broadcast_to(
        np.arange(n_bunches)[None, :, None], tofs.shape
    )[valid]
    in_win = (tof_of_hit >= TOF_MIN) & (tof_of_hit < TOF_MAX)

    return {
        "data": data,
        "n_trains": n_trains,
        "n_bunches": n_bunches,
        "max_hits":  max_hits,
        "tof_of_hit":   tof_of_hit,
        "train_of_hit": train_of_hit,
        "bunch_of_hit": bunch_of_hit,
        "in_win": in_win,
    }


runs = {_label_for(i, p): load_and_prepare(p) for i, p in enumerate(FILES)}

for label, run in runs.items():
    n_t = run["n_trains"]
    n_b = run["n_bunches"]
    print(f"{label}:")
    print(f"  shape    : ({n_t}, {n_b}, {run['max_hits']})  "
          f"= {n_t * n_b} shots")
    print(f"  hits     : {run['tof_of_hit'].size} non-zero, "
          f"{int(run['in_win'].sum())} in window")

## Plot 1 — average spectrum per file (overlay)

Mean counts per shot across all (train, bunch) shots, one curve per
file. Useful for spotting gain drifts, peak shifts, or relative
intensity differences.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for label, run in runs.items():
    total_counts, _ = np.histogram(run["tof_of_hit"], bins=tof_edges)
    mean_spec = total_counts / (run["n_trains"] * run["n_bunches"])
    ax.plot(tof_cents, mean_spec, lw=1.2,
            label=f"{label}  ({run['n_trains'] * run['n_bunches']} shots)")
ax.set_xlabel("TOF (100 ps)")
ax.set_ylabel("Counts per shot")
ax.set_title(f"Average {TOF_KEY} spectrum  "
             f"({TOF_MIN:.0f}-{TOF_MAX:.0f} 100 ps)")
ax.set_xlim(tof_cents[0], tof_cents[-1])
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Plot 2 — mean spectrum vs bunch index

One 2D map per file (wrapped at `MAX_COLS` columns), followed by an
overlay of the integrated mean hits per bunch across files.

In [ ]:
n_files = len(runs)

# 2D maps per file.
fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=False, sharex=True,
)
mean_by_bunch_runs = {}
for k, (label, run) in enumerate(runs.items()):
    bunch_edges = np.arange(run["n_bunches"] + 1)
    H_bunch, _, _ = np.histogram2d(
        run["bunch_of_hit"], run["tof_of_hit"],
        bins=[bunch_edges, tof_edges],
    )
    mean_by_bunch = H_bunch / run["n_trains"]
    mean_by_bunch_runs[label] = mean_by_bunch

    ax = ax_at(k, 0)
    norm = (mcolors.LogNorm(vmin=max(np.nanmin(mean_by_bunch), 1e-6),
                            vmax=np.nanmax(mean_by_bunch))
            if LOG_MAP else None)
    im = ax.pcolormesh(tof_cents, np.arange(run["n_bunches"]),
                       mean_by_bunch, cmap="inferno",
                       shading="auto", norm=norm)
    fig.colorbar(im, ax=ax, label="Mean counts per shot")
    ax.set_xlabel("TOF (100 ps)")
    ax.set_ylabel("Bunch index")
    ax.set_title(f"{label}")
hide_unused()
fig.suptitle(f"Mean {TOF_KEY} spectrum vs bunch index")
plt.show()

# Overlay of integrated hits per bunch.
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for label, mean_by_bunch in mean_by_bunch_runs.items():
    total = mean_by_bunch.sum(axis=1)
    ax.plot(np.arange(total.size), total, marker="o", ms=3, lw=1.2,
            label=label)
ax.set_xlabel("Bunch index")
ax.set_ylabel("Mean hits per shot")
ax.set_title("Integrated hits per bunch")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Plot 3 — mean spectrum vs train index

Same approach with train index in place of bunch. One 2D map per file
(wrapped), plus an overlay of integrated hits per train across files
(raw + Savitzky-Golay smoothed).

In [ ]:
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=False, sharex=True,
)
mean_by_train_runs = {}
for k, (label, run) in enumerate(runs.items()):
    train_edges = np.arange(run["n_trains"] + 1)
    H_train, _, _ = np.histogram2d(
        run["train_of_hit"], run["tof_of_hit"],
        bins=[train_edges, tof_edges],
    )
    mean_by_train = H_train / run["n_bunches"]
    mean_by_train_runs[label] = (mean_by_train, run["data"].tID)

    ax = ax_at(k, 0)
    norm = (mcolors.LogNorm(vmin=max(np.nanmin(mean_by_train), 1e-6),
                            vmax=np.nanmax(mean_by_train))
            if LOG_MAP else None)
    im = ax.pcolormesh(tof_cents, run["data"].tID, mean_by_train,
                       cmap="inferno", shading="auto", norm=norm)
    fig.colorbar(im, ax=ax, label="Mean counts per shot")
    ax.set_xlabel("TOF (100 ps)")
    ax.set_ylabel("Train ID")
    ax.set_title(label)
hide_unused()
fig.suptitle(f"Mean {TOF_KEY} spectrum vs train index")
plt.show()

# Overlay of integrated hits per train (raw + smoothed).
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for label, (mean_by_train, tID) in mean_by_train_runs.items():
    total = mean_by_train.sum(axis=1)
    line, = ax.plot(tID, total, marker="o", ms=2, lw=0,
                    label=f"{label} (raw)")
    if total.size >= 9:
        smoothed = savgol_filter(total, 9, 3)
        ax.plot(tID, smoothed, lw=1.5, color=line.get_color(),
                label=f"{label} (savgol)")
ax.set_xlabel("Train ID")
ax.set_ylabel("Mean hits per shot")
ax.set_title("Integrated hits per train")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=max(1, n_files // 2))
plt.show()

## Plot 4 — GMD vs per-shot hit count

One hist2d per file (per-shot scatter, log shot density) overlaid
with a linear fit. Files wrap past `MAX_COLS`. A separate overlay
plots the GMD-percentile-binned mean ± std with each file's linear
fit superimposed.

In [ ]:
def _per_shot_counts(run):
    """Per-shot hit count inside the TOF window for one run."""
    in_win = run["in_win"]
    shot_id = (run["train_of_hit"][in_win] * run["n_bunches"]
               + run["bunch_of_hit"][in_win])
    n_shots = run["n_trains"] * run["n_bunches"]
    return np.bincount(shot_id, minlength=n_shots).astype(np.float64)


# Pre-compute the binning + fit per file.
gmd_info = {}
for label, run in runs.items():
    counts = _per_shot_counts(run)
    gmd_flat = run["data"].gmd.ravel()
    good = np.isfinite(gmd_flat)
    x = gmd_flat[good]
    y = counts[good]

    gmd_edges = np.percentile(x, np.linspace(0, 100, N_GMD_BINS + 1))
    gmd_cents, bool_ar = arb_bool_ar(gmd_edges, x)
    mean_hits, std_hits, n_per_bin = bin_and_average(bool_ar, y)
    mean_gmd, _, _ = bin_and_average(bool_ar, x)

    slope, intercept = np.polyfit(mean_gmd, mean_hits, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    gmd_info[label] = {
        "x": x, "y": y,
        "gmd_edges": gmd_edges,
        "mean_gmd": mean_gmd,
        "mean_hits": mean_hits,
        "std_hits": std_hits,
        "n_per_bin": n_per_bin,
        "bool_ar": bool_ar,
        "good": good,
        "slope": slope,
        "intercept": intercept,
        "r": r,
    }

n_files = len(runs)

# Per-file hist2d + fit.
fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=False, sharex=False,
)
for k, (label, info) in enumerate(gmd_info.items()):
    ax = ax_at(k, 0)
    x, y = info["x"], info["y"]
    h = ax.hist2d(x, y,
                  bins=[info["gmd_edges"], np.arange(10) - 0.5],
                  cmap="viridis", cmin=1)
    fig.colorbar(h[3], ax=ax, label="Shots per bin")
    x_line = np.linspace(info["mean_gmd"].min(),
                         info["mean_gmd"].max(), 100)
    ax.plot(x_line, info["slope"] * x_line + info["intercept"],
            color="white", lw=1.5,
            label=f"fit: y = {info['slope']:.3g}·x + "
                  f"{info['intercept']:.3g}\n"
                  f"Pearson r = {info['r']:.3f}")
    ax.set_xlabel("GMD (uJ)")
    ax.set_ylabel(f"Hits per shot")
    ax.set_title(f"{label}  ({x.size} shots)")
    ax.legend(loc="upper left", framealpha=0.85, fontsize=8)
hide_unused()
fig.suptitle(f"GMD vs {TOF_KEY} hits ({TOF_MIN:.0f}-{TOF_MAX:.0f} 100 ps)")
plt.show()

# Overlay of binned mean +/- std with per-file fit lines.
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for label, info in gmd_info.items():
    line = ax.errorbar(
        info["mean_gmd"], info["mean_hits"],
        yerr=info["std_hits"], fmt="o-", capsize=3, lw=1.2,
        label=f"{label} (slope={info['slope']:.3g}, r={info['r']:.3f})",
    )
    col = line[0].get_color()
    x_line = np.linspace(info["mean_gmd"].min(),
                         info["mean_gmd"].max(), 100)
    ax.plot(x_line, info["slope"] * x_line + info["intercept"],
            color=col, ls="--", lw=1.2)
ax.set_xlabel("GMD (uJ)")
ax.set_ylabel(f"Mean hits per shot ({TOF_MIN:.0f}-{TOF_MAX:.0f} 100 ps)")
ax.set_title(f"Binned mean GMD vs {TOF_KEY} hits  ({N_GMD_BINS} percentile bins)")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)
plt.show()

## Plot 5 — average TOF spectrum per GMD bin (one panel per file)

For each file, use the same GMD percentile binning as plot 4. Look
up each in-window hit's parent shot's GMD bin and histogram by
`(gmd_bin, tof)` in a single `np.histogram2d` call — so the
per-shot spectra cube is never materialised. Each row of the
resulting `(n_gmd_bins, n_tof_bins)` array is divided by the shots
in that bin to get mean counts per shot.

One panel per file, lines coloured by GMD bin centre. Panels wrap
at `MAX_COLS`.

In [ ]:
def _spec_by_gmd(run, info):
    """(n_gmd_bins, n_tof_bins) of mean counts per shot per GMD bin."""
    n_shots = run["n_trains"] * run["n_bunches"]
    bool_ar = info["bool_ar"]
    n_gmd_bins = bool_ar.shape[0]

    gmd_bin_of_shot = np.full(n_shots, -1, dtype=int)
    good_idx = np.flatnonzero(info["good"])
    for i in range(n_gmd_bins):
        gmd_bin_of_shot[good_idx[bool_ar[i]]] = i

    in_win = run["in_win"]
    tof_in   = run["tof_of_hit"][in_win]
    shot_id  = (run["train_of_hit"][in_win] * run["n_bunches"]
                + run["bunch_of_hit"][in_win])
    hit_gmd_bin = gmd_bin_of_shot[shot_id]
    keep = hit_gmd_bin >= 0

    H, _, _ = np.histogram2d(
        hit_gmd_bin[keep], tof_in[keep],
        bins=[np.arange(n_gmd_bins + 1) - 0.5, tof_edges],
    )
    return H / info["n_per_bin"][:, None]


n_files = len(runs)
spec_by_gmd_runs = {
    label: _spec_by_gmd(run, gmd_info[label])
    for label, run in runs.items()
}

fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=True, sharex=True,
)
for k, (label, run) in enumerate(runs.items()):
    info = gmd_info[label]
    spec = spec_by_gmd_runs[label]
    mean_gmd = info["mean_gmd"]
    norm = Normalize(vmin=float(mean_gmd[0]), vmax=float(mean_gmd[-1]))

    ax = ax_at(k, 0)
    for cent, s in zip(mean_gmd, spec):
        ax.plot(tof_cents, s, color=plt.cm.viridis(norm(cent)), lw=1.2)
    sm = ScalarMappable(norm=norm, cmap="viridis"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Mean GMD per bin (uJ)")
    ax.set_xlabel("TOF (100 ps)")
    ax.set_ylabel("Mean counts per shot")
    ax.set_title(f"{label}  ({len(mean_gmd)} GMD bins)")
    ax.set_xlim(tof_cents[0], tof_cents[-1])
    if YLIM_SPEC_BY_GMD is not None:
        ax.set_ylim(*YLIM_SPEC_BY_GMD)
    ax.grid(alpha=0.3)
hide_unused()
fig.suptitle(f"Average {TOF_KEY} spectrum per GMD percentile bin")
plt.show()

## Plot 6 — same spectrum, GMD-normalised

Each spectrum from plot 5 is divided by its bin's mean GMD, giving
counts per shot per uJ. Useful for comparing the intrinsic
(fluence-independent) line shape across GMD bins / files.

In [ ]:
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=True, sharex=True,
)
for k, (label, run) in enumerate(runs.items()):
    info = gmd_info[label]
    spec = spec_by_gmd_runs[label] / info["mean_gmd"][:, None]
    mean_gmd = info["mean_gmd"]
    norm = Normalize(vmin=float(mean_gmd[0]), vmax=float(mean_gmd[-1]))

    ax = ax_at(k, 0)
    for cent, s in zip(mean_gmd, spec):
        ax.plot(tof_cents, s, color=plt.cm.viridis(norm(cent)), lw=1.2)
    sm = ScalarMappable(norm=norm, cmap="viridis"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Mean GMD per bin (uJ)")
    ax.set_xlabel("TOF (100 ps)")
    ax.set_ylabel("Mean counts per shot per uJ")
    ax.set_title(f"{label}  ({len(mean_gmd)} GMD bins)")
    ax.set_xlim(tof_cents[0], tof_cents[-1])
    ax.grid(alpha=0.3)
hide_unused()
fig.suptitle(f"Average {TOF_KEY} spectrum per GMD bin, GMD-normalised")
plt.show()